# 04a（US）控制变量回归（OLS/Logit + HAC）

目标：在 `03_ganzhi_effect_analysis` 的样本内分组统计基础上，加入日历控制变量（`weekday/month/year`），并用 HAC / Newey-West 稳健标准误估计组效应。

输入：
- `data/clean_us/market_ganzhi_{ts_code}.csv.gz`

输出（写入 `data/clean_us/robustness/`，每个 `ts_code × group_col × target` 一份）：
- `controls_{ts_code}_{group_col}_{target}.csv`

说明：
- `p/q`：相对基准水平的系数检验（语义弱，尤其在 60 甲子）
- `p/q_effect`：更推荐使用的“组 vs 全样本边际均值差”的 contrast 检验


In [1]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy import stats

warnings.filterwarnings('ignore')

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# plots
import matplotlib.pyplot as plt
import seaborn as sns



from matplotlib import font_manager, rcParams


def set_chinese_font() -> None:
    # Best-effort Chinese font setup for Matplotlib.
    # - Tries common Linux CJK fonts
    # - On WSL, also tries registering Windows fonts under /mnt/c/Windows/Fonts

    candidates = [
        'Noto Sans CJK SC',
        'Noto Sans CJK TC',
        'Noto Sans CJK JP',
        'Source Han Sans SC',
        'WenQuanYi Micro Hei',
        'Microsoft YaHei',
        'SimHei',
        'SimSun',
        'Arial Unicode MS',
    ]

    # WSL fallback: register Windows fonts if available
    try:
        from pathlib import Path as _Path

        win_dir = _Path('/mnt/c/Windows/Fonts')
        win_files = [
            win_dir / 'msyh.ttc',
            win_dir / 'msyhbd.ttc',
            win_dir / 'simhei.ttf',
            win_dir / 'simsun.ttc',
            win_dir / 'arialuni.ttf',
        ]

        extra_names: list[str] = []
        addfont = getattr(font_manager.fontManager, 'addfont', None)
        for fp in win_files:
            try:
                if fp.is_file():
                    if callable(addfont):
                        addfont(str(fp))
                    extra_names.append(font_manager.FontProperties(fname=str(fp)).get_name())
            except Exception:
                continue

        candidates = [n for n in extra_names if n] + candidates
    except Exception:
        pass

    for name in candidates:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            rcParams['font.sans-serif'] = [name]
            rcParams['axes.unicode_minus'] = False
            print(f'Using Chinese font: {name}')
            return
        except Exception:
            continue

    rcParams['axes.unicode_minus'] = False
    print(
        'Warning: no Chinese font found; Chinese labels may not render. '
        'If you are on WSL, ensure Windows fonts are accessible under /mnt/c/Windows/Fonts; '
        'or install Noto CJK fonts in Linux (e.g., fonts-noto-cjk).'
    )


set_chinese_font()

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean_us'
ROBUST_DIR = CLEAN_DIR / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Config (edit as needed)
# =====================
TS_CODES = ['spx', 'ndq', 'ndx', 'dji']

# Control-variable models to run
GROUP_COLS = ['stem', 'branch']
RUN_GANZHI_DAY_MODELS = True

# Permutation test (raw series; global)
RANDOM_SEED = 20260213
N_PERM = 1000

# Permutation on controls-only residuals (more conservative)
N_PERM_RESID = 2000

# HAC / Newey-West (time-series dependence)
HAC_MAXLAGS = 5
HAC_MAXLAGS_SWEEP = [0, 1, 3, 5, 10]

# Block bootstrap (controls-only residuals)
N_BOOT = 1000
BOOT_BLOCK_LEN = 10

# Walk-forward
TRAIN_YEARS = 5
TRAIN_YEARS_SWEEP = [3, 5, 7]

STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')

# Phase 2: Li-Chun year element interaction (day_group × year_element)
RUN_PHASE2_INTERACTIONS = True
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_P_THRESHOLD = 0.10
PHASE2_GATE_Q = 0.10
PHASE2_GATE_MIN_SHARE = 0.75


def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)


def load_market_ganzhi(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}. Run notebooks_US 01-03 first.')
    df = pd.read_csv(path, compression='gzip', dtype={'trade_date': str})
    if df.empty:
        raise ValueError(f'Empty data: {path}')
    return df


def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['trade_date'], format='%Y%m%d')
    out['date'] = dt
    out['weekday'] = dt.dt.weekday.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['year'] = dt.dt.year.astype(int)
    return out


def set_category_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'stem' in out.columns:
        out['stem'] = pd.Categorical(out['stem'], categories=STEMS_ORDER, ordered=True)
    if 'branch' in out.columns:
        out['branch'] = pd.Categorical(out['branch'], categories=BRANCHES_ORDER, ordered=True)
    return out




Using Chinese font: Microsoft YaHei
PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支


## 1) 含控制变量的回归（方向/收益）


In [2]:
@dataclass
class ControlModelSpec:
    group_col: str
    target: str  # 'up' or 'ret_1d'
    formula: str


def fit_control_model(df: pd.DataFrame, spec: ControlModelSpec):
    if spec.target == 'up':
        model = smf.glm(formula=spec.formula, data=df, family=sm.families.Binomial())
        return model.fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS}, maxiter=200, disp=0)

    if spec.target == 'ret_1d':
        model = smf.ols(formula=spec.formula, data=df)
        return model.fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})

    raise ValueError(f'Unknown target: {spec.target}')


def extract_group_terms(res, group_col: str) -> pd.DataFrame:
    params = res.params
    pvals = res.pvalues

    rows = []
    for name in params.index:
        if name.startswith(f'C({group_col})['):
            rows.append(
                {
                    'term': name,
                    'coef': float(params[name]),
                    'p_value': float(pvals[name]) if np.isfinite(pvals[name]) else np.nan,
                }
            )

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    p = out['p_value'].to_numpy(dtype=float)
    ok = np.isfinite(p)
    q = np.full_like(p, np.nan, dtype=float)
    if ok.sum() > 0:
        q[ok] = fdr_bh(p[ok])
    out['q_value'] = q

    out['group_col'] = group_col
    out['group_value'] = out['term'].str.extract(r'\[T\.(.*)\]')[0]

    return out.sort_values(['q_value', 'p_value'])


def marginal_mean_by_group(res, df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    # Marginal mean: set group_col to each value and average predictions over observed controls
    values = pd.Series(df[group_col].dropna().unique()).astype(str).sort_values().tolist()

    if isinstance(df[group_col].dtype, pd.CategoricalDtype):
        values = [str(v) for v in df[group_col].dtype.categories if v in set(df[group_col].dropna())]

    base_pred = res.predict(df)
    base_mean = float(np.nanmean(base_pred))

    records = []
    for v in values:
        tmp = df.copy()
        if isinstance(df[group_col].dtype, pd.CategoricalDtype):
            tmp[group_col] = pd.Categorical(
                [v] * len(tmp),
                categories=df[group_col].dtype.categories,
                ordered=df[group_col].dtype.ordered,
            )
        else:
            tmp[group_col] = v
        pred = res.predict(tmp)
        mu = float(np.nanmean(pred))
        records.append(
            {
                'group_col': group_col,
                'group_value': v,
                'marginal_mean': mu,
                'marginal_effect_vs_all': mu - base_mean,
            }
        )

    out = pd.DataFrame.from_records(records)
    out['marginal_mean_all'] = base_mean
    return out


def marginal_effect_tests_vs_all(
    res,
    df: pd.DataFrame,
    group_col: str,
    target: str,
    group_values: list[str],
) -> pd.DataFrame:
    values = [str(v) for v in group_values]

    if target == 'ret_1d':
        design_info = res.model.data.design_info
        x_base = res.model.exog
        x_base_mean = np.nanmean(x_base, axis=0)

        records = []
        for v in values:
            tmp = df.copy()
            if isinstance(df[group_col].dtype, pd.CategoricalDtype):
                tmp[group_col] = pd.Categorical(
                    [v] * len(tmp),
                    categories=df[group_col].dtype.categories,
                    ordered=df[group_col].dtype.ordered,
                )
            else:
                tmp[group_col] = v

            x_v = build_design_matrices([design_info], tmp, return_type='dataframe')[0].to_numpy()
            l = np.nanmean(x_v, axis=0) - x_base_mean
            t = res.t_test(l)

            se = float(np.asarray(t.sd).ravel()[0])
            z = float(np.asarray(t.tvalue).ravel()[0])
            p = float(np.asarray(t.pvalue).ravel()[0])
            records.append({'group_value': v, 'se_effect': se, 'z_effect': z, 'p_value_effect': p})

        out = pd.DataFrame.from_records(records)
        p = out['p_value_effect'].to_numpy(dtype=float)
        ok = np.isfinite(p)
        q = np.full_like(p, np.nan, dtype=float)
        if ok.sum() > 0:
            q[ok] = fdr_bh(p[ok])
        out['q_value_effect'] = q
        return out

    if target == 'up':
        beta = res.params.to_numpy()
        cov = res.cov_params().to_numpy()

        design_info = res.model.data.design_info
        x_base = res.model.exog
        eta_base = x_base @ beta
        mu_base = res.model.family.link.inverse(eta_base)
        base_mean = float(np.nanmean(mu_base))
        w_base = res.model.family.link.inverse_deriv(eta_base)
        g_base = np.nanmean(w_base[:, None] * x_base, axis=0)

        records = []
        for v in values:
            tmp = df.copy()
            if isinstance(df[group_col].dtype, pd.CategoricalDtype):
                tmp[group_col] = pd.Categorical(
                    [v] * len(tmp),
                    categories=df[group_col].dtype.categories,
                    ordered=df[group_col].dtype.ordered,
                )
            else:
                tmp[group_col] = v

            x_v = build_design_matrices([design_info], tmp, return_type='dataframe')[0].to_numpy()
            eta_v = x_v @ beta
            mu_v = res.model.family.link.inverse(eta_v)
            mean_v = float(np.nanmean(mu_v))

            w_v = res.model.family.link.inverse_deriv(eta_v)
            g_v = np.nanmean(w_v[:, None] * x_v, axis=0)

            effect = mean_v - base_mean
            g_diff = g_v - g_base
            var = float(g_diff @ cov @ g_diff)

            if not np.isfinite(var) or var <= 0:
                se = np.nan
                z = np.nan
                p = np.nan
            else:
                se = float(np.sqrt(var))
                z = float(effect / se)
                p = float(2.0 * stats.norm.sf(abs(z)))

            records.append({'group_value': v, 'se_effect': se, 'z_effect': z, 'p_value_effect': p})

        out = pd.DataFrame.from_records(records)
        p = out['p_value_effect'].to_numpy(dtype=float)
        ok = np.isfinite(p)
        q = np.full_like(p, np.nan, dtype=float)
        if ok.sum() > 0:
            q[ok] = fdr_bh(p[ok])
        out['q_value_effect'] = q
        return out

    raise ValueError(f'Unknown target: {target}')


def run_controls_for_ts_code(ts_code: str, group_cols: list[str]) -> None:
    df = load_market_ganzhi(ts_code)
    df = add_calendar_controls(df)
    df = set_category_orders(df)

    df = df.dropna(subset=['ret_1d', 'up'])

    print('\n===', ts_code, '===')
    print('n_days =', len(df), '| range =', df['trade_date'].min(), '->', df['trade_date'].max())

    for group_col in group_cols:
        if group_col not in df.columns:
            print('skip missing group col:', group_col)
            continue

        if not isinstance(df[group_col].dtype, pd.CategoricalDtype):
            df[group_col] = df[group_col].astype('category')

        specs = [
            ControlModelSpec(
                group_col=group_col,
                target='up',
                formula=f"up ~ C({group_col}) + C(weekday) + C(month) + C(year)",
            ),
            ControlModelSpec(
                group_col=group_col,
                target='ret_1d',
                formula=f"ret_1d ~ C({group_col}) + C(weekday) + C(month) + C(year)",
            ),
        ]

        for spec in specs:
            try:
                res = fit_control_model(df, spec)
            except Exception as exc:
                print(f'[{ts_code}] model failed:', spec.target, spec.group_col, exc)
                continue

            coef_tbl = extract_group_terms(res, group_col)
            mm_tbl = marginal_mean_by_group(res, df, group_col)
            eff_tbl = marginal_effect_tests_vs_all(res, df, group_col, spec.target, mm_tbl['group_value'].tolist())

            out = mm_tbl.merge(eff_tbl, on='group_value', how='left')
            out = out.merge(coef_tbl[['group_value', 'coef', 'p_value', 'q_value']], on='group_value', how='left')
            out.insert(0, 'ts_code', ts_code)
            out.insert(1, 'target', spec.target)

            out_path = ROBUST_DIR / f'controls_{ts_code}_{group_col}_{spec.target}.csv'
            out.to_csv(out_path, index=False)

            top = out.sort_values(['q_value_effect', 'p_value_effect']).head(10)
            print(f'[{group_col}][{spec.target}] saved:', out_path)
            display_cols = [
                'group_value',
                'marginal_effect_vs_all',
                'p_value_effect',
                'q_value_effect',
                'coef',
                'p_value',
                'q_value',
                'marginal_mean',
            ]
            print(top[display_cols].to_string(index=False))


cols = GROUP_COLS + (['ganzhi_day'] if RUN_GANZHI_DAY_MODELS else [])
for ts_code in TS_CODES:
    run_controls_for_ts_code(ts_code, cols)





=== spx ===
n_days = 4053 | range = 20100105 -> 20260213
[stem][up] saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean_us\robustness\controls_spx_stem_up.csv
group_value  marginal_effect_vs_all  p_value_effect  q_value_effect      coef  p_value  q_value  marginal_mean
          辛                0.047067        0.040580        0.405801  0.261974 0.061504 0.553537       0.593576
          壬               -0.039959        0.090099        0.450493 -0.094333 0.505569 0.839068       0.506549
          庚                0.022676        0.329896        0.914380  0.160672 0.262819 0.839068       0.569184
          乙                0.019187        0.414787        0.914380  0.146305 0.318772 0.839068       0.565695
          甲               -0.016659        0.475330        0.914380       NaN      NaN      NaN       0.529850
          癸               -0.009721        0.681098        0.914380  0.028172 0.839068 0.839068       0.536788
          戊               -0.009105        0.695871        0.914380  0.